# FLUX.1-Kontext-dev API Server on Google Colab

Notebook này cho phép bạn:
- Cài đặt dependencies
- Tải model FLUX.1-Kontext-dev
- Chạy API server
- Test API trực tiếp

**Yêu cầu:** GPU runtime (T4 hoặc cao hơn)

## 1. Cài đặt Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate safetensors
!pip install -q fastapi uvicorn pydantic pillow
!pip install -q huggingface-hub sentencepiece protobuf
!pip install -q pyngrok  # Để expose server ra internet

print("✅ Dependencies installed!")

## 2. Kiểm tra GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU detected! Please enable GPU runtime.")
    print("Go to: Runtime > Change runtime type > GPU")

## 3. Đăng nhập Hugging Face

Lấy token tại: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login
import os

# Nhập token của bạn
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxx"  # @param {type:"string"}

login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
print("✅ Logged in to Hugging Face!")

## 4. Tải Model

In [ ]:
from diffusers import FluxKontextPipeline
import torch

print("Loading FLUX.1-Kontext-dev model...")
print("This may take 5-10 minutes for the first time.")

pipe = FluxKontextPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-Kontext-dev",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)

# Enable memory optimization for Colab GPUs
pipe.enable_model_cpu_offload()

print("✅ Model loaded successfully!")

## 5. Test Generation trực tiếp

In [ ]:
from IPython.display import display

# Test text-to-image
prompt = "A cute cat wearing sunglasses, photorealistic"  # @param {type:"string"}

print(f"Generating: {prompt}")

result = pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    num_inference_steps=50,
    guidance_scale=2.5
)

image = result.images[0]
display(image)

# Save
image.save("test_output.png")
print("✅ Image saved to test_output.png")

## 6. Tạo API Server

In [ ]:
%%writefile api_server_colab.py
import os
import io
import base64
import torch
import requests as http_requests
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel, Field
from typing import Optional
from PIL import Image
from diffusers import FluxKontextPipeline

app = FastAPI(title="FLUX.1-Kontext-dev API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

pipe = None

class GenerateRequest(BaseModel):
    prompt: str
    width: int = 1024
    height: int = 1024
    num_inference_steps: int = 50
    guidance_scale: float = 2.5
    seed: Optional[int] = None
    output_format: str = "base64"

class EditRequest(BaseModel):
    prompt: str
    input_image: Optional[str] = None
    image_url: Optional[str] = None
    num_inference_steps: int = 50
    guidance_scale: float = 2.5
    seed: Optional[int] = None
    output_format: str = "base64"

class GenerationResponse(BaseModel):
    success: bool
    message: str
    image_base64: Optional[str] = None
    seed: Optional[int] = None

def load_model():
    global pipe
    print("Loading model...")
    pipe = FluxKontextPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-Kontext-dev",
        torch_dtype=torch.bfloat16,
        token=os.environ.get("HF_TOKEN")
    )
    pipe.enable_model_cpu_offload()
    print("Model loaded!")

@app.on_event("startup")
async def startup():
    load_model()

@app.get("/health")
async def health():
    return {"status": "healthy", "model_loaded": pipe is not None}

@app.post("/generate", response_model=GenerationResponse)
async def generate(request: GenerateRequest):
    if pipe is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    generator = None
    seed = request.seed
    if seed is not None:
        generator = torch.Generator(device="cpu").manual_seed(seed)
    else:
        seed = torch.randint(0, 2**32, (1,)).item()
    
    result = pipe(
        prompt=request.prompt,
        height=request.height,
        width=request.width,
        num_inference_steps=request.num_inference_steps,
        guidance_scale=request.guidance_scale,
        generator=generator
    )
    
    image = result.images[0]
    
    if request.output_format == "base64":
        buffered = io.BytesIO()
        image.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode()
        return GenerationResponse(
            success=True,
            message="Image generated",
            image_base64=img_base64,
            seed=seed
        )
    else:
        buffered = io.BytesIO()
        image.save(buffered, format="PNG")
        return Response(content=buffered.getvalue(), media_type="image/png")

@app.post("/edit", response_model=GenerationResponse)
async def edit(request: EditRequest):
    if pipe is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    if not request.input_image and not request.image_url:
        raise HTTPException(status_code=400, detail="input_image or image_url required")
    
    # Load input image
    if request.input_image:
        img_data = base64.b64decode(request.input_image)
        input_img = Image.open(io.BytesIO(img_data)).convert("RGB")
    else:
        response = http_requests.get(request.image_url, timeout=30)
        input_img = Image.open(io.BytesIO(response.content)).convert("RGB")
    
    generator = None
    seed = request.seed
    if seed is not None:
        generator = torch.Generator(device="cpu").manual_seed(seed)
    else:
        seed = torch.randint(0, 2**32, (1,)).item()
    
    result = pipe(
        image=input_img,
        prompt=request.prompt,
        num_inference_steps=request.num_inference_steps,
        guidance_scale=request.guidance_scale,
        generator=generator
    )
    
    image = result.images[0]
    
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    img_base64 = base64.b64encode(buffered.getvalue()).decode()
    
    return GenerationResponse(
        success=True,
        message="Image edited",
        image_base64=img_base64,
        seed=seed
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

## 7. Chạy Server với Ngrok (Public URL)

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import threading

# Apply nest_asyncio for Colab
nest_asyncio.apply()

# Ngrok auth token (get from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = ""  # @param {type:"string"}

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print("="*50)
print("FLUX.1-Kontext-dev API Server")
print("="*50)
print(f"Public URL: {public_url}")
print("="*50)
print("\nClients can connect using the URL above")
print("\nEndpoints:")
print(f"  Health: {public_url}/health")
print(f"  Generate: {public_url}/generate")
print(f"  Edit: {public_url}/edit")
print("="*50)

In [ ]:
# Run server (this will block)
!python api_server_colab.py

## 8. Test API từ Client

Chạy cell này trong một notebook khác hoặc từ máy local:

In [ ]:
import requests
import base64
from IPython.display import display, Image as IPImage
from PIL import Image
import io

# Thay bằng URL ngrok của bạn
API_URL = "https://xxxx-xx-xx-xx-xx.ngrok-free.app"  # @param {type:"string"}

# Health check
response = requests.get(f"{API_URL}/health")
print("Health:", response.json())

In [ ]:
# Generate image
prompt = "A futuristic city with flying cars"  # @param {type:"string"}

response = requests.post(
    f"{API_URL}/generate",
    json={
        "prompt": prompt,
        "width": 1024,
        "height": 1024,
        "num_inference_steps": 50,
        "output_format": "base64"
    },
    timeout=300
)

if response.status_code == 200:
    result = response.json()
    img_data = base64.b64decode(result["image_base64"])
    img = Image.open(io.BytesIO(img_data))
    display(img)
    img.save("generated.png")
    print(f"Seed: {result.get('seed')}")
else:
    print(f"Error: {response.text}")

In [ ]:
# Edit image
# Upload image or use URL
image_url = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"  # @param {type:"string"}
edit_prompt = "Add aurora borealis to the sky"  # @param {type:"string"}

response = requests.post(
    f"{API_URL}/edit",
    json={
        "prompt": edit_prompt,
        "image_url": image_url,
        "num_inference_steps": 50,
        "output_format": "base64"
    },
    timeout=300
)

if response.status_code == 200:
    result = response.json()
    img_data = base64.b64decode(result["image_base64"])
    img = Image.open(io.BytesIO(img_data))
    display(img)
    img.save("edited.png")
else:
    print(f"Error: {response.text}")

## 9. Utility Functions

In [ ]:
# Function để generate dễ dàng
def generate_image(prompt, width=1024, height=1024, steps=50, seed=None):
    """Generate image from text prompt"""
    response = requests.post(
        f"{API_URL}/generate",
        json={
            "prompt": prompt,
            "width": width,
            "height": height,
            "num_inference_steps": steps,
            "seed": seed,
            "output_format": "base64"
        },
        timeout=300
    )
    
    if response.status_code == 200:
        result = response.json()
        img_data = base64.b64decode(result["image_base64"])
        return Image.open(io.BytesIO(img_data)), result.get("seed")
    else:
        raise Exception(f"Error: {response.text}")

def edit_image(prompt, image_path=None, image_url=None, steps=50):
    """Edit image with text prompt"""
    payload = {
        "prompt": prompt,
        "num_inference_steps": steps,
        "output_format": "base64"
    }
    
    if image_path:
        with open(image_path, "rb") as f:
            payload["input_image"] = base64.b64encode(f.read()).decode()
    elif image_url:
        payload["image_url"] = image_url
    else:
        raise ValueError("Provide image_path or image_url")
    
    response = requests.post(
        f"{API_URL}/edit",
        json=payload,
        timeout=300
    )
    
    if response.status_code == 200:
        result = response.json()
        img_data = base64.b64decode(result["image_base64"])
        return Image.open(io.BytesIO(img_data))
    else:
        raise Exception(f"Error: {response.text}")

print("✅ Utility functions loaded!")
print("Usage:")
print('  img, seed = generate_image("A cat")')
print('  img = edit_image("Add snow", image_url="https://...")')

In [ ]:
# Example usage
img, seed = generate_image("A beautiful sunset over ocean")
display(img)
print(f"Seed: {seed}")

## 10. Cleanup

In [ ]:
# Stop ngrok tunnel
ngrok.kill()
print("✅ Ngrok tunnel closed")